# Mova — local training (Apple Silicon)

Train the clinical **freezing-of-gait** model and the activity-recognition model on your Mac (MPS), with no GPU cluster.

**Kernel:** pick the `.venv312` (Python 3.12) interpreter, then run the cells top-to-bottom.

In [ ]:
import os, sys, pathlib
import numpy as np, torch

here = pathlib.Path.cwd()
REPO = here if (here / 'src').exists() else here.parent
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
os.environ.setdefault('PYTORCH_ENABLE_MPS_FALLBACK', '1')

device = 'mps' if torch.backends.mps.is_available() else ('cuda' if torch.cuda.is_available() else 'cpu')
print('repo :', REPO)
print('torch:', torch.__version__, '| device:', device)

## 1 · Data — freezing-of-gait windows

In [ ]:
from mova.train.data import make_dataloaders

fog_loaders, fog_vocabs = make_dataloaders(mode='fog', batch_size=128, num_workers=0, pin_memory=False)
for split, dl in fog_loaders.items():
    y = dl.dataset.labels()
    print(f'{split:>5}: {len(dl.dataset):>6} windows | freeze {int((y==1).sum()):>5} ({(y==1).mean():.1%})')

## 2 · Train the FoG model
A compact LIMU-BERT encoder + binary head, ~12 epochs on MPS.

In [ ]:
import pytorch_lightning as pl
from mova.train.module import MovaLitModule

pl.seed_everything(1337, workers=True)
encoder_cfg = dict(in_channels=6, max_len=200, hidden=192, n_layers=4, n_heads=6,
                   ff_mult=4, dropout=0.1, use_conditioning=True)
EPOCHS = 12
steps = len(fog_loaders['train']) * EPOCHS
fog_model = MovaLitModule(encoder_cfg=encoder_cfg, task='fog', num_classes=2,
                          n_placements=fog_vocabs.n_placements, n_datasets=fog_vocabs.n_datasets,
                          lr=3e-4, weight_decay=0.05, warmup_ratio=0.1, max_steps=steps)
trainer = pl.Trainer(accelerator='auto', devices=1, precision='32-true', max_epochs=EPOCHS,
                     gradient_clip_val=1.0, logger=False, enable_checkpointing=False, log_every_n_steps=20)
trainer.fit(fog_model, fog_loaders['train'], fog_loaders['val'])

## 3 · Evaluate (held-out, subject-disjoint test set)

In [ ]:
from mova.eval.clinical import collect_predictions, fog_metrics

y, pred, score = collect_predictions(fog_model, fog_loaders['test'], device)
m = fog_metrics(y, pred, score)
for k, v in m.as_dict().items():
    print(f'{k:>18}: {v:.4f}' if isinstance(v, float) else f'{k:>18}: {v}')

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay, RocCurveDisplay

fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ConfusionMatrixDisplay.from_predictions(y, pred, display_labels=['no_freeze', 'freeze'], ax=ax[0], colorbar=False)
ax[0].set_title('FoG confusion · test')
RocCurveDisplay.from_predictions(y, score, ax=ax[1])
ax[1].set_title(f'ROC · AUROC = {m.auroc:.3f}')
plt.tight_layout(); plt.show()

## 4 · Activity recognition (HAR)
Same encoder, 37-class head.

In [ ]:
from sklearn.metrics import accuracy_score, f1_score

har_loaders, har_vocabs = make_dataloaders(mode='har', batch_size=256, num_workers=0, pin_memory=False)
H_EPOCHS = 6
har_model = MovaLitModule(encoder_cfg=encoder_cfg, task='har', num_classes=len(har_vocabs.har_label),
                          n_placements=har_vocabs.n_placements, n_datasets=har_vocabs.n_datasets,
                          lr=3e-4, max_steps=len(har_loaders['train']) * H_EPOCHS)
pl.Trainer(accelerator='auto', devices=1, precision='32-true', max_epochs=H_EPOCHS,
           logger=False, enable_checkpointing=False).fit(har_model, har_loaders['train'], har_loaders['val'])

har_model.eval().to(device)
ys, ps = [], []
with torch.no_grad():
    for b in har_loaders['test']:
        h = har_model.encoder(b['x'].to(device), b['placement'].to(device), b['dataset'].to(device))
        ps.append(har_model.head(har_model.encoder.pool(h)).argmax(1).cpu().numpy())
        ys.append(b['label'].numpy())
ys, ps = np.concatenate(ys), np.concatenate(ps)
mf1 = f1_score(ys, ps, average='macro')
acc = accuracy_score(ys, ps)
print(f'HAR test · accuracy {acc:.3f} · macro-F1 {mf1:.3f}')

---
**Stronger numbers:** self-supervised pretrain first (`bash scripts/train_local.sh ssl`), then pass the checkpoint via `finetune.pretrained_ckpt`. The CLI path (`scripts/train.py trainer=local`) mirrors this notebook.